# Week 3: Set up an agent for simple fiber route planning

In **03_01**, we shortened the Mombasa-Taveta alignment and downloaded nearby infrastructure. Now we will give those saved layers to a coding agent and ask it to compare simple routes between the **two endpoints of the shortened corridor**. Our fiber route destination is the cut point from not Taveta.

Keep this first application simple: find the shortest available path along each infrastructure layer, account for small gaps explicitly, and estimate cost (using a simplified cost per meter).

By the end, you should be able to give an agent a clear task, explain how lines become a routing graph, inspect disconnected routes, and check its distance and cost calculations. The agent will write and run the routing implementation; this notebook prepares its inputs and provides the method.

## 1. Reuse the data from 03_01

Open the course repository as your workspace. No further downloads should be needed here.

| File saved by 03_01 | Role here |
|---|---|
| `route1_short.gpkg` | Baseline and exact start/end locations |
| `roads.gpkg` | Road corridor candidate |
| `power_lines.gpkg` | Power corridor candidate |
| `rail_lines.gpkg` | Rail corridor candidate |
| `settlements.gpkg` | Map context only |

03_01 does not save the bounding box polygon, currently. 

If you think you need it, you can reconstruct the same rectangle from the shortened route with a 3,000 m margin (or you can use a buffered polygon for the route). If you changed that margin in 03_01, change it below too. 

If you want to put the files in a new folder, so everything is kept in a consistent place, feel free to do that. 

## 2. Decide on endpoints 

Every candidate must connect the same two points. 

The map uses local data only. 

If you find gaps e.g., for infrastructure that never reaches the endpoint, you can create a final new linestring path to connect it (often called the "final hop"). The gap tolerance can initially be up to 500 m (so we can build a new line for anything under this distance). 

## 3. Give the agent standing instructions

Try write your demo **`AGENTS.md`** file by giving it routing instructions  (named in uppercase by convention). 

You should describe the inputs, shortest-path method, connector limits, cost equation, and expected outputs.

Ask your coding agent to read that file explicitly before starting. You do not need a separate custom agent definition for this lesson. The notebook variables are not automatically visible to a chat agent.

The instructions apply to this routing exercise. Keep the original input GeoPackages unchanged.

## 4. A simple shortest-path method

Ask the agent to build one undirected shortest-path using a combination of the best roads, power, and rail lines. 

These should be possible fiber corridors, so traffic one-way restrictions do not control direction. 

This is a simplified geometric network, not a vehicle-routing network.

1. Keep valid line geometries, explode multipart lines, remove duplicate geometry, and clip to the study rectangle. Report dropped features.
2. Split lines at intersections, then make each consecutive coordinate pair an edge with its actual geometry and `length_m`. Nodes are coordinate pairs. Splitting at every vertex also handles closed lines without losing their lengths. For this lesson, within-layer crossings are assumed connectable in 2D; flag bridge/tunnel/grade-separation uncertainty.
3. Inspect connected components. Connect nearby components only under the gap rule below. Never silently keep only the largest component.
4. Attach the exact start and end to their nearest points on the network, splitting the touched edges at those points. Add the endpoint connectors and include their lengths. If either connector exceeds its limit, report that candidate as unavailable.
5. Use Dijkstra's algorithm with `length_m` as the weight. Reconstruct the route from the ordered edge geometries, including connectors. Report no path if the endpoints remain disconnected.

NetworkX provides the search once the agent has prepared the graph:

```python
path_nodes = nx.shortest_path(G, source=start_node, target=end_node,
                              weight="length_m", method="dijkstra")
```


### Handling lines that do not connect

Use **connector lines**, so students can see and measure every inferred gap. Do not move the original data or increase tolerances until a route appears.

- Start with a **50 m maximum gap** between different connected components. Find their nearest points, split the affected edges, and insert a straight connector only if its entire geometry stays within the study area. Mark it `gap_connector`. Repeat from the smallest eligible gap, recomputing components, until no eligible gaps remain.
- Use a separate **500 m maximum** for each connection from the fixed start/end to the nearest network point. Mark these `endpoint_connector`. Attach a point already on a line as a node without creating a zero-length edge.
- Keep inferred connectors separate from observed infrastructure. Report their count, total length, and longest length for each route. Every connector is hypothetical new construction, not proof of a missing OSM feature.
- If no path exists, record `no_path`; if an endpoint is too far away, record `endpoint_too_far`. Leave route length and cost blank for unavailable candidates. Do not replace them with straight lines.

These thresholds are classroom choices. A string of small connectors can still be implausible, so inspect the full map and the total connector distance.

## 5. Estimate costs with one multiplication

Use **USD 10 per metre** for every baseline, infrastructure segment, and connector. This is an invented teaching value, not a price quotation. We leave out terrain, permissions, equipment, and infrastructure-specific pricing at this stage.

**Route cost (USD) = total route length (m) * cost per metre (USD/m).**

For example, a 2,000 m route including 100 m of connectors costs USD 20,000 at this rate. The connectors are already included in the 2,000 m; do not add them twice. With the same positive rate for every route, the shortest available route is also the cheapest under this model.

## 6. Hand the task to the agent

In the VS Code agents window, you can then paste your prompt into your coding agent. For example:

> Read `AGENTS.md`. Use the local layers saved by 03_01 to compare the shortened baseline with road, power, and rail candidates between the saved endpoints. Create a graph of all network routes (road, rail, power, etc.)  use the connector rules in this notebook and AGENTS.md.
>
> Write a reproducible Python routing script, run it, inspect failures, and validate its results. Find a shortest path by distance for each infrastructure layer separately. Use explicit, bounded connectors for disconnected lines and endpoint access. Report unavailable candidates honestly. Do not download more data or change the thresholds automatically.
>
> Calculate total distance and estimated USD cost using the saved cost per metre, including all connector distance exactly once. Save route and connector geometries, a comparison CSV, a map, and a short report under `outputs/week3_routing/`. Include all four comparison rows, even when a candidate is unavailable. State which available route is shortest and cheapest under this simple model and explain the limitations.

Watch the agent **inspect, implement, run, check, revise, report**. The algorithm is specified here; the agent still has to turn imperfect spatial data into a working, checked analysis.

## 7. Inspect the results

Ask for these outputs:

| Output | Contents |
|---|---|
| `routes.gpkg` | Baseline and successful candidate routes |
| `connectors.gpkg` | Used gap/endpoint connectors with route and type; omit if none are used and state this in the report |
| `route_comparison.csv` | Four rows: baseline, roads, power, rail; status, length, connector distance/count/maximum, rate, cost |
| `route_map.png` | Inputs, routes, fixed endpoints, and clearly distinguished connectors, with OSM attribution |
| `routing_report.md` | Method, run command, settings, validation, failures, and interpretation |

Lengths must use metres in EPSG:32737. Successful routes must reach both exact endpoints, remain continuous and within the study area, and have geometry length matching the sum of their traversed edge lengths. Connector limits apply to individual connectors, not their combined length. Costs must equal the unrounded length multiplied by the rate; round only for display.

## 8. Challenge one assumption

Ask the agent:

> Repeat the analysis by relaxing the gap tolerance from 500 m to 250 m. Keep endpoints, endpoint connector limit, and cost per metre unchanged. Save the rerun in a separate subfolder, preserving the first results. Which candidates remain connected, and how do their distances and costs change?

Discuss:

1. Which connections came from mapped lines and which were invented by the model?
2. Did an unavailable candidate indicate missing data, a real gap, or a restrictive threshold?
3. Why does doubling a common cost per metre double costs without changing their ranking?
4. Which route would you investigate further, and what evidence would you need?

These are exploratory fiber corridors. Spatial proximity does not establish permission to use a road, railway, or power corridor.